In [23]:
import sklearn
import seaborn
import matplotlib
from pandas import *
import janitor
from IPython.display import display

In [24]:
df_antecedents_familiaux = read_csv("../../csv/Antecedents_familiaux.csv").clean_names(strip_underscores=True)
df_centres_soins = read_csv("../../csv/Centres_soins.csv").clean_names(strip_underscores=True)
df_consultations = read_csv("../../csv/Consultations.csv").clean_names(strip_underscores=True)
df_expression_genique = read_csv("../../csv/Expression_genique.csv").clean_names(strip_underscores=True)
df_historique_glycemique = read_csv("../../csv/Historique_glycemique.csv").clean_names(strip_underscores=True)
df_hygiene_vie = read_csv("../../csv/Hygiene_vie.csv").clean_names(strip_underscores=True)
df_patients_centres = read_csv("../../csv/Patients_centres.csv").clean_names(strip_underscores=True)
df_patients = read_csv("../../csv/Patients.csv").clean_names(strip_underscores=True)
df_traitements = read_csv("../../csv/Traitements.csv").clean_names(strip_underscores=True)

In [25]:
df_dict = {
    "antecedents_familiaux": df_antecedents_familiaux,
    "centres_soins": df_centres_soins,
    "consultations": df_consultations,
    "expression_genique": df_expression_genique,
    "historique_glycemique": df_historique_glycemique,
    "hygiene_vie": df_hygiene_vie,
    "patients_centres": df_patients_centres,
    "patients": df_patients,
    "traitements": df_traitements,
}

## 2. Exploration préliminaire

### 2.0 Check initiaux

In [26]:
display(df_patients.shape)
display(df_patients["id_patient"].unique())
# le dataframe principal a 1500 patients tous unique

(1500, 9)

<StringArray>
[ 'P001',  'P002',  'P003',  'P004',  'P005',  'P006',  'P007',  'P008',
  'P009',  'P010',
 ...
 'P1491', 'P1492', 'P1493', 'P1494', 'P1495', 'P1496', 'P1497', 'P1498',
 'P1499', 'P1500']
Length: 1500, dtype: str

### 2.1 Aperçu de chaque table

In [27]:
def describe(df, stats=[], decimals=1):
    d = df.describe()
    if stats:
      d = concat([d, df.reindex(d.columns, axis = 1).agg(stats)])
    return d.round(decimals)

In [28]:
def apercu(df, cols=None, max_modalites=20):
    if cols is not None:
        df = df[[cols] if isinstance(cols, str) else cols]
        max_modalites = None
    display(df.head().style.set_caption("head"))
    display(describe(df).style.set_caption("describe").format(precision=1))
    for c in df.columns:
        if max_modalites is None or df[c].nunique() <= max_modalites:
            display(df[c].value_counts(dropna=False).to_frame().style.set_caption(f"value_counts : {c}"))

#### Valeurs manquantes

In [29]:
for df_name, df in df_dict.items():
  df_total_na = df.isna().sum() 
  df_postive_na_only = df.isna().sum()[df.isna().sum() != 0]
  if df_postive_na_only.shape[0] > 0:
    print(f'### NA DANS {df_name} ###\n')
    print(df_postive_na_only)
    print("\n------------------------------------\n")

# la table patients a 4 sexe manquants et 6 présence de corticoides manquantes
# la table traitements a 149 dosage manquants, et 149 date de début manquante.
# les dates de fin manquantes sont attendues car des traitements sont encore en cours

### NA DANS patients ###

sexe           4
corticoides    6
dtype: int64

------------------------------------

### NA DANS traitements ###

date_de_debut     149
date_de_fin      1343
dosage            149
dtype: int64

------------------------------------



#### Patients

In [ ]:
apercu(df_patients, "glycemie_mg_dl", 2000)

In [31]:
apercu(df_patients)
# liste des anomlies trouvées :
# un age de 278 ans
# des valeurs sexe mal formatée et manquantes

,id_patient,age_ans,sexe,poids_kg,taille_cm,groupe_sanguin,glycemie_mg_dl,insuline_µui_ml,corticoides
0,P001,37,F,48.000000,157.100000,O+,74.700000,9.100000,Non
1,P002,47,M,58.600000,175.100000,O+,78.000000,6.300000,Non
2,P003,44,F,95.000000,172.200000,A+,95.900000,12.700000,Non
3,P004,52,f,70.200000,159.200000,O+,96.600000,6.400000,Non
4,P005,58,M,99.300000,172.200000,A+,104.500000,13.900000,Oui


,age_ans,poids_kg,taille_cm,glycemie_mg_dl,insuline_µui_ml
count,1500.0,1500.0,1500.0,1500.0,1500.0
mean,50.9,72.5,168.8,93.4,9.4
std,17.9,15.5,10.7,17.4,4.2
min,18.0,41.2,145.0,62.0,2.0
25%,39.0,60.9,161.1,83.2,6.4
50%,51.0,71.2,168.6,91.5,9.1
75%,63.0,82.5,175.4,100.5,12.1
max,278.0,128.2,200.0,200.9,25.1


,count
sexe,
F,923
M,561
nan,4
femme,3
homme,2
Femme,2
f,1
m,1
H,1


,count
groupe_sanguin,
O+,593
A+,426
B+,184
AB+,107
O-,82
A-,61
AB-,33
B-,14


,count
corticoides,
Non,1364
Oui,128
nan,6
na,1
Na,1


In [32]:
display(df_patients[df_patients["glycemie_mg_dl"] >= 126].shape)
display(df_patients[df_patients["glycemie_mg_dl"] < 126].shape)
# on a affaire a un dataset très déséquilibré 1450 classe négative contre 50 classes positives

(50, 9)

(1450, 9)

#### Antécédents familiaux

In [33]:
apercu(df_antecedents_familiaux)

,id_patient,lien_de_parente,pathologie
0,P001,Aucun,Aucun antécédent connu
1,P002,Grand-parent,Hypertension
2,P003,Aucun,Aucun antécédent connu
3,P004,Aucun,Aucun antécédent connu
4,P005,Aucun,Aucun antécédent connu


,id_patient,lien_de_parente,pathologie
count,1662,1662,1662
unique,1500,6,5
top,P023,Aucun,Aucun antécédent connu
freq,3,1222,1222


,count
lien_de_parente,
Aucun,1222
Grand-parent,97
Frère,93
Sœur,89
Mère,83
Père,78


,count
pathologie,
Aucun antécédent connu,1222
Diabète de type 2,146
Hypertension,117
Obésité,89
Maladie cardiovasculaire,88


In [48]:
df_antecedents_familiaux[df_antecedents_familiaux.duplicated(subset=["id_patient"]) == True]

,id_patient,lien_de_parente,pathologie
10,P010,Sœur,Hypertension
12,P011,Grand-parent,Obésité
25,P023,Sœur,Obésité
26,P023,Grand-parent,Hypertension
37,P033,Mère,Hypertension
...,...,...,...
1638,P1481,Grand-parent,Hypertension
1649,P1491,Mère,Obésité
1651,P1492,Grand-parent,Diabète de type 2
1652,P1492,Père,Hypertension


#### Centres de soins

In [35]:
apercu(df_centres_soins)

,id_centre,nom,region,type
0,C01,Centre de soins synthétique 01,Bruxelles-Capitale,Hôpital universitaire
1,C02,Centre de soins synthétique 02,Wallonie - Brabant wallon,Hôpital général
2,C03,Centre de soins synthétique 03,Wallonie - Hainaut,Maison médicale
3,C04,Centre de soins synthétique 04,Wallonie - Liège,Centre de diabétologie
4,C05,Centre de soins synthétique 05,Wallonie - Namur,Hôpital universitaire


,id_centre,nom,region,type
count,24,24,24,24
unique,24,24,11,4
top,C01,Centre de soins synthétique 01,Bruxelles-Capitale,Hôpital universitaire
freq,1,1,3,6


,count
region,
Bruxelles-Capitale,3
Wallonie - Brabant wallon,3
Wallonie - Hainaut,2
Wallonie - Liège,2
Wallonie - Namur,2
Wallonie - Luxembourg,2
Flandre - Anvers,2
Flandre - Brabant flamand,2
Flandre - Flandre-Orientale,2


,count
type,
Hôpital universitaire,6
Hôpital général,6
Maison médicale,6
Centre de diabétologie,6


#### Consultations

In [36]:
apercu(df_consultations)

,id_patient,date,type_de_consultation,medecin_service,motif
0,P001,2025-09-06,Médecine générale,Médecine générale,Prévention
1,P001,2025-11-07,Bilan,Médecine interne,Contrôle biologique
2,P002,2025-06-09,Médecine générale,Médecine générale,Adaptation du traitement
3,P002,2024-10-08,Médecine générale,Médecine générale,Adaptation du traitement
4,P003,2025-12-01,Endocrinologie,Endocrinologie,Suivi HbA1c


,id_patient,date,type_de_consultation,medecin_service,motif
count,3609,3609,3609,3609,3609
unique,1500,895,6,6,9
top,P154,2024-11-15,Médecine générale,Médecine générale,Suivi HbA1c
freq,7,12,893,893,823


,count
type_de_consultation,
Médecine générale,893
Suivi diabétologique,824
Endocrinologie,801
Diététique,784
Prévention,161
Bilan,146


,count
medecin_service,
Médecine générale,893
Service de diabétologie,824
Endocrinologie,801
Diététicien·ne,784
Consultation prévention,161
Médecine interne,146


,count
motif,
Suivi HbA1c,823
Contrôle glycémique,785
Conseils hygiéno-diététiques,779
Adaptation du traitement,777
Prévention,156
Contrôle biologique,143
Bilan annuel,126
Suivi métabolique,11
Gestion du poids,9


#### Expression génique

In [37]:
apercu(df_expression_genique)

,id_patient,tcf7l2,pparg,kcnj11,abcc8,slc30a8,cdkal1,cdkn2a,igf2bp2,hhex,fto,irs1,gck,gckr,mtnr1b,kcnq1,adcy5,glis3,jazf1,hnf1a
0,P001,6.913000,8.176000,7.006000,6.749000,7.366000,6.390000,7.122000,7.004000,6.358000,7.577000,7.676000,8.190000,8.061000,6.617000,6.404000,6.835000,6.230000,6.469000,7.577000
1,P002,7.097000,8.360000,6.127000,6.883000,6.810000,6.532000,7.433000,6.893000,5.936000,7.433000,7.658000,8.130000,7.993000,6.325000,6.211000,7.188000,6.264000,6.820000,7.006000
2,P003,7.258000,7.794000,6.232000,6.916000,7.472000,6.630000,7.002000,6.846000,6.438000,7.793000,7.923000,8.547000,7.909000,6.631000,6.934000,7.405000,6.149000,6.649000,7.747000
3,P004,6.548000,7.869000,6.963000,6.884000,6.208000,7.060000,7.722000,6.362000,6.868000,7.460000,7.208000,8.210000,8.310000,5.919000,6.739000,7.489000,6.104000,6.743000,6.637000
4,P005,7.493000,7.935000,5.800000,7.279000,7.354000,7.564000,8.323000,7.266000,6.028000,8.341000,7.032000,8.128000,7.738000,7.083000,7.204000,6.848000,6.406000,6.986000,7.112000


,tcf7l2,pparg,kcnj11,abcc8,slc30a8,cdkal1,cdkn2a,igf2bp2,hhex,fto,irs1,gck,gckr,mtnr1b,kcnq1,adcy5,glis3,jazf1,hnf1a
count,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0
mean,7.2,8.0,6.4,6.8,7.1,6.9,7.6,7.0,6.5,7.9,7.4,8.2,7.8,6.6,6.7,7.1,6.3,6.9,7.5
std,0.5,0.5,0.4,0.4,0.5,0.4,0.4,0.4,0.4,0.5,0.5,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4
min,5.7,6.6,5.0,5.4,5.5,5.3,6.5,5.6,5.0,6.2,5.9,6.3,6.4,5.3,5.3,5.4,4.9,5.5,6.0
25%,6.9,7.7,6.1,6.5,6.8,6.6,7.3,6.7,6.2,7.6,7.1,7.9,7.5,6.3,6.4,6.8,6.0,6.6,7.2
50%,7.2,8.0,6.4,6.8,7.1,6.9,7.6,7.0,6.5,7.9,7.4,8.2,7.8,6.6,6.7,7.1,6.3,6.9,7.5
75%,7.5,8.3,6.7,7.1,7.4,7.2,7.9,7.3,6.8,8.2,7.7,8.5,8.1,6.9,7.0,7.4,6.6,7.2,7.8
max,8.7,9.5,7.8,8.3,9.0,8.2,9.0,8.3,8.0,9.4,9.0,9.6,9.2,8.1,8.0,8.5,7.7,8.3,9.0


#### Historique glycémique

In [38]:
apercu(df_historique_glycemique)

,id_patient,date_de_mesure,glycemie_mg_dl,poids_kg,tension_arterielle_mmhg,hba1c_%
0,P001,2024-11-01,89.700000,50.300000,97/65,7.080000
1,P001,2025-03-21,69.900000,47.700000,117/62,6.410000
2,P001,2025-03-31,76.900000,44.100000,125/62,6.510000
3,P001,2025-04-15,75.700000,44.300000,105/64,6.530000
4,P001,2025-05-12,71.300000,45.200000,105/72,5.970000


,glycemie_mg_dl,poids_kg,hba1c_%
count,11565.0,11565.0,11565.0
mean,93.1,72.5,7.1
std,18.6,15.8,0.6
min,58.0,40.0,5.5
25%,81.6,60.9,6.7
50%,91.4,71.4,7.0
75%,101.4,82.8,7.4
max,214.5,131.5,11.1


#### Hygiène de vie

In [39]:
apercu(df_hygiene_vie)

,id_patient,stress_0_10,niveau_de_stress,sommeil_h_nuit,activite_physique_min_sem,niveau_dactivite,fruits_et_legumes_portions_jour,boissons_sucrees_portions_sem,fast_food_repas_sem,score_alimentation_0_10,qualite_de_lalimentation,tabagisme,alcool_unites_sem
0,P001,6,Modéré,7.600000,197.000000,Élevée,2.900000,3,2.200000,4.800000,Moyennement équilibrée,Non,3.600000
1,P002,5,Modéré,6.200000,166.000000,Élevée,4.700000,5,1.500000,6.000000,Moyennement équilibrée,Non,9.700000
2,P003,5,Modéré,7.000000,118.000000,Modérée,2.200000,3,1.700000,1.500000,Peu équilibrée,Non,0.000000
3,P004,6,Modéré,7.000000,156.000000,Élevée,2.200000,3,1.000000,3.200000,Peu équilibrée,Non,2.000000
4,P005,7,Élevé,6.200000,193.000000,Élevée,1.300000,1,1.500000,2.400000,Peu équilibrée,Oui,14.300000


,stress_0_10,sommeil_h_nuit,activite_physique_min_sem,fruits_et_legumes_portions_jour,boissons_sucrees_portions_sem,fast_food_repas_sem,score_alimentation_0_10,alcool_unites_sem
count,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0,1500.0
mean,5.1,7.2,153.6,3.6,2.9,1.2,4.0,4.3
std,2.4,1.8,88.5,1.6,2.4,1.0,1.8,3.8
min,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,4.0,6.5,83.0,2.5,1.0,0.3,2.8,1.0
50%,5.0,7.2,146.4,3.6,3.0,1.0,4.0,3.8
75%,7.0,7.9,211.9,4.8,4.0,1.9,5.1,6.7
max,10.0,64.0,450.0,8.9,12.0,5.0,9.5,17.9


,count
stress_0_10,
5,270
6,251
4,209
7,169
3,152
8,125
2,103
0,60
9,60


,count
niveau_de_stress,
Modéré,730
Élevé,400
Faible,370


,count
niveau_dactivite,
Élevée,731
Modérée,456
Faible,312
faible,1


,count
boissons_sucrees_portions_sem,
0,322
2,233
3,197
1,193
4,185
5,149
6,97
7,64
8,31


,count
qualite_de_lalimentation,
Peu équilibrée,755
Moyennement équilibrée,665
Équilibrée,80


,count
tabagisme,
Non,1211
Oui,289


#### Patients / centres

In [40]:
apercu(df_patients_centres)

,id_patient,id_centre
0,P001,C21
1,P002,C14
2,P003,C18
3,P004,C09
4,P005,C22


,id_patient,id_centre
count,1500,1500
unique,1500,24
top,P001,C09
freq,1,84


#### Traitements

In [41]:
apercu(df_traitements)

,id_patient,medicament,date_de_debut,date_de_fin,dosage,categorie_medicament
0,P001,Aucun traitement,nan,nan,nan,Aucun traitement
1,P002,Metformine,2025-02-10,nan,850 mg 2×/j,Antidiabétiques
2,P002,Cétirizine,2025-12-22,2026-01-03,10 mg 1×/j,Antihistaminiques
3,P003,Metformine,2023-09-28,2025-12-10,500 mg 1×/j,Antidiabétiques
4,P004,Metformine,2023-12-31,nan,500 mg 1×/j,Antidiabétiques


,id_patient,medicament,date_de_debut,date_de_fin,dosage,categorie_medicament
count,2432,2432,2283,1089,2283,2432
unique,1500,16,915,505,14,6
top,P016,Metformine,2025-01-09,2026-01-20,500 mg 1×/j,Antidiabétiques
freq,3,1345,9,9,643,1375


,count
medicament,
Metformine,1345
Amoxicilline,300
Aucun traitement,149
Azithromycine,140
Cétirizine,75
Loratadine,75
Doxycycline,70
Sertraline,50
Prednisone,46


,count
dosage,
500 mg 1×/j,643
500 mg 2×/j,447
850 mg 2×/j,395
1 g 2×/j,300
10 mg 1×/j,190
nan,149
100 mg 2×/j,70
50 mg 1×/j,50
5–20 mg/j,46


,count
categorie_medicament,
Antidiabétiques,1375
Antibiotiques,510
Antihistaminiques,150
Aucun traitement,149
Corticoïdes,128
Antidépresseurs,120


### 2.3 Analyse des NAN

In [42]:
for df_name, df in df_dict.items():
  df_total_na = df.isna().sum() 
  df_postive_na_only = df.isna().sum()[df.isna().sum() != 0]
  if df_postive_na_only.shape[0] > 0:
    print(f'### NA DANS {df_name} ###\n')
    print(df_postive_na_only)
    print("\n------------------------------------\n")

# dans nos potentielles futures features il n'y a que les traitements qui disposent
# de valeurs manquates. 149 dosage manquants, et 149 date de début manquante.
# les dates de fin manquantes sont attendues car des traitements sont encore en cours

### NA DANS patients ###

sexe           4
corticoides    6
dtype: int64

------------------------------------

### NA DANS traitements ###

date_de_debut     149
date_de_fin      1343
dosage            149
dtype: int64

------------------------------------



### 2.4 Valeurs uniques

In [ ]:
# on analyse les valeurs unqiue des variables

for df_name, df in df_dict.items():
  print(f"\n### VALEUR UNIQUE DANS {df_name}  ###\n")
  print(Series(data={c: df[c].unique() for c in df.columns}))


### VALEUR UNIQUE DANS antecedents_familiaux  ###

id_patient         [P001, P002, P003, P004, P005, P006, P007, P00...
lien_de_parente       [Aucun, Grand-parent, Frère, Sœur, Mère, Père]
pathologie         [Aucun antécédent connu, Hypertension, Obésité...
dtype: object

### VALEUR UNIQUE DANS centres_soins  ###

id_centre    [C01, C02, C03, C04, C05, C06, C07, C08, C09, ...
nom          [Centre de soins synthétique 01, Centre de soi...
region       [Bruxelles-Capitale, Wallonie - Brabant wallon...
type         [Hôpital universitaire, Hôpital général, Maiso...
dtype: object

### VALEUR UNIQUE DANS consultations  ###

id_patient              [P001, P002, P003, P004, P005, P006, P007, P00...
date                    [2025-09-06, 2025-11-07, 2025-06-09, 2024-10-0...
type_de_consultation    [Médecine générale, Bilan, Endocrinologie, Dié...
medecin_service         [Médecine générale, Médecine interne, Endocrin...
motif                   [Prévention, Contrôle biologique, Adaptation d...
dty